# Attention 메커니즘 비교: MHA vs GQA vs MLA — Attention Map + KV Cache Footprint

**KCC 2026 튜토리얼 (보조 노트북)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tteon/kcc2026-tutorial/blob/main/kvcache_attention_mha_gqa_mla.ipynb)

같은 입력에 대해 **MHA · GQA · MLA** 세 어텐션 구조의
1. **Attention Map** (head 별 패턴 차이) 과
2. **KV Cache Footprint** (토큰당 메모리 · 컨텍스트 증가 시 폭발)

을 나란히 보여줍니다. 메인 튜토리얼의 *KVCache 재사용*이 **왜 중요한지**를 아키텍처 관점에서 보강합니다.

> **권장 런타임: Colab A100 (40GB)**. gpt2 / Qwen2.5-1.5B 는 어디서나 돌아가고, MLA(DeepSeek-V2-Lite, bf16 ~31GB)만 A100 이 필요합니다.
> MLA 셀은 건너뛰어도 KV footprint 비교(수치/그래프)는 그대로 동작합니다.


## 0. 환경 설정

`output_attentions=True` 로 어텐션 가중치를 직접 뽑으려면 **`attn_implementation="eager"`** 가 필수입니다.
(SDPA / FlashAttention 커널은 softmax 가중치를 밖으로 돌려주지 않아 `attentions` 가 `None` 이 됩니다.)

In [ ]:
# 0-1. 패키지 설치
# 이 보조 노트북은 torch/transformers 계열 의존성이 필요합니다.
# 메인 튜토리얼용 run.sh만 실행한 로컬 환경이라면 이 셀을 실행하세요.
%pip install -q "transformers>=4.44" torch accelerate matplotlib sentencepiece
print("installed")

In [ ]:
# 0-2. import & 디바이스
import gc, math
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if (DEVICE == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32
print("device =", DEVICE, "| dtype =", DTYPE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

## 1. 세 구조의 핵심 차이

| | Query heads | **KV heads** | KV cache (토큰당) | 핵심 아이디어 |
|---|---|---|---|---|
| **MHA** (Multi-Head) | H | **H** | `2·L·H·d` | head 마다 독립 K/V. 표현력 최대, 메모리 최대 |
| **GQA** (Grouped-Query) | H | **H/g** | `2·L·(H/g)·d` | 여러 Q head 가 KV head 하나를 공유 → KV `g` 배 절감 |
| **MLA** (Multi-head Latent) | H | **압축 latent** | `L·(r + d_rope)` | K/V 를 저랭크 latent `c_KV` 로 압축 저장 → 극단적 절감 |

- `L`=레이어 수, `H`=어텐션 head 수, `d`=head dim, `g`=group 크기, `r`=`kv_lora_rank`.
- **Attention Map** 으로 보는 것: MHA 는 head 간 패턴이 제각각, GQA 는 *같은 KV group* head 들이 거시 패턴을 공유, MLA 는 latent 압축에도 head 별 패턴이 살아있음.
- **KV Footprint** 로 보는 것: 같은 7B급 모델이라도 MHA→GQA→MLA 로 갈수록 토큰당 캐시가 급감 → long context 에서 VRAM 폭발을 막음.


In [ ]:
# 1-1. 직관 그림: Query head 는 많아도, 저장하는 KV head 수가 줄어들면 캐시가 작아진다.
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))

def draw_panel(ax, title, q_heads, kv_groups, note):
    ax.axis("off")
    ax.set_title(title, fontsize=13, weight="bold")
    for i in range(q_heads):
        ax.add_patch(plt.Circle((0.7 + i*0.42, 2.0), 0.13, facecolor="#9ecae1", edgecolor="#333"))
        ax.text(0.7 + i*0.42, 2.0, "Q", ha="center", va="center", fontsize=8)
    for j, (x, width, label, color) in enumerate(kv_groups):
        ax.add_patch(plt.Rectangle((x, 0.8), width, 0.38, facecolor=color, edgecolor="#333"))
        ax.text(x + width/2, 0.99, label, ha="center", va="center", fontsize=9)
    ax.text(0.2, 0.25, note, fontsize=10)
    ax.set_xlim(0, 4.6); ax.set_ylim(0, 2.5)

draw_panel(
    axes[0], "MHA", 8,
    [(0.45+i*0.42, 0.32, "KV", "#fdae6b") for i in range(8)],
    "Q head 마다 K/V 저장\n메모리 최대"
)
draw_panel(
    axes[1], "GQA", 8,
    [(0.55, 1.45, "KV group 0", "#a1d99b"), (2.35, 1.45, "KV group 1", "#a1d99b")],
    "여러 Q head 가 KV 공유\nMHA 대비 약 group 배 절감"
)
draw_panel(
    axes[2], "MLA", 8,
    [(1.0, 2.4, "compressed latent KV", "#c7b9ff")],
    "K/V 를 latent 로 압축 저장\nlong context 에 유리"
)
fig.suptitle("어텐션 구조 차이를 KV cache 관점에서 보기", fontsize=14, weight="bold")
plt.tight_layout(); plt.show()

## 2. ⚠️ 입력 데이터: `hash_ids` 는 token id 가 아니다

메인 튜토리얼 trace 의 `hash_ids`(예: `[0, 4, 1, 5, 16, 1642]`)는 **16-토큰 블록의 prefix 해시 ID** 입니다.
**token id 가 아니므로** 모델에 그대로 넣으면 의미 없는 어텐션이 나옵니다.

대신 어텐션은 **실제 텍스트를 토크나이즈**해서 봅니다. 단, trace 의 핵심인 *prefix 공유* 를 살리려고
**앞부분이 반복되는 문장**을 입력으로 씁니다 → 어텐션 맵에서 prefix 로 향하는 패턴(induction / attention sink)이 보이고,
이는 곧 메인 튜토리얼의 *KVCache 재사용* 과 직결됩니다.

In [ ]:
# 2-1. 데모 입력: 앞부분이 반복되는 문장 (prefix 공유 모사)
DEMO_TEXT = "The cat sat on the mat. The cat sat on the"
# 반복되는 "The cat sat on the" 구간이 prefix. 두 번째 등장 토큰들이 첫 등장으로 어텐션을 보내는지 관찰.

def clean_tok(t: str) -> str:
    # 토크나이저별 공백/개행 마커 정리 (gpt2 'Ġ', sentencepiece '▁' 등)
    return (t.replace("Ġ", " ").replace("▁", " ").replace("Ċ", "\\n")).strip() or t

## 3. 공통 헬퍼: 모델 로드 · 어텐션 추출 · 시각화

모델은 **한 번에 하나만** 올리고, 끝나면 GPU 메모리를 비웁니다(셋을 동시에 올리지 않음).

In [ ]:
# 3-1. 로드 / 어텐션 추출 / 메모리 해제
def load_model(model_id):
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=DTYPE,
        attn_implementation="eager",   # ← output_attentions 의 전제조건
        device_map=DEVICE,
        trust_remote_code=True,        # DeepSeek-V2 커스텀 코드용
    ).eval()
    return tok, model

@torch.no_grad()
def get_attentions(tok, model, text):
    enc = tok(text, return_tensors="pt").to(model.device)
    out = model(**enc, output_attentions=True, use_cache=False)
    if out.attentions is None:
        raise RuntimeError("attentions=None — attn_implementation='eager' 인지 확인하세요.")
    toks = tok.convert_ids_to_tokens(enc["input_ids"][0])
    # attentions: list[L] of (batch, H, q, kv) → list[L] of (H, q, kv) numpy
    atts = [a[0].float().cpu().numpy() for a in out.attentions]
    return toks, atts

def kv_group_size(cfg):
    nh  = cfg.num_attention_heads
    nkv = getattr(cfg, "num_key_value_heads", nh) or nh
    return max(1, nh // nkv)

def free(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# 3-2. head grid 시각화 (한 레이어의 여러 head 를 나란히)
def plot_heads(atts, layer, heads, toks, title, group_size=None):
    labels = [clean_tok(t) for t in toks]
    n = len(heads); cols = min(n, 4); rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3.6 * cols, 3.2 * rows), squeeze=False)
    axes = axes.reshape(-1)
    for i, h in enumerate(heads):
        ax = axes[i]
        im = ax.imshow(atts[layer][h], cmap="viridis", aspect="auto")
        ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=90, fontsize=7)
        ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=7)
        grp = "" if group_size is None else f"  ·  KV grp {h // group_size}"
        ax.set_title(f"head {h}{grp}", fontsize=10)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    for j in range(len(heads), len(axes)):
        axes[j].axis("off")
    fig.suptitle(title, fontsize=13)
    plt.tight_layout(); plt.show()

# 어텐션 행(row)은 query 토큰, 열(col)은 key 토큰. causal 이라 상삼각은 0.
# 각 row 의 합 = 1 (softmax). 밝을수록 그 key 에 강하게 attend.

## 4. MHA — head 마다 제각각 (gpt2)

`gpt2` 는 전통적 MHA(12 layer, 12 head, head_dim 64). KV head = Query head 라 **모든 head 가 독립적인 K/V** 를 가집니다.
중간 레이어에서 head 들을 나란히 보면 패턴이 서로 다릅니다 — 어떤 head 는 직전 토큰, 어떤 head 는 첫 토큰(sink), 어떤 head 는 반복 prefix 로 attend.

In [ ]:
# 4-1. MHA: gpt2
tok_mha, model_mha = load_model("gpt2")
cfg = model_mha.config
print(f"MHA gpt2 | layers={cfg.num_hidden_layers} heads={cfg.num_attention_heads} "
      f"kv_heads={getattr(cfg,'num_key_value_heads',cfg.num_attention_heads)} group={kv_group_size(cfg)}")

toks_mha, atts_mha = get_attentions(tok_mha, model_mha, DEMO_TEXT)
print("tokens:", [clean_tok(t) for t in toks_mha])

L = cfg.num_hidden_layers
plot_heads(atts_mha, layer=L // 2, heads=[0, 1, 2, 3], toks=toks_mha,
           title=f"MHA (gpt2) — layer {L//2}: head 별 패턴이 제각각")

## 5. GQA — 같은 KV group 은 닮는다 (Qwen2.5-1.5B)

GQA 는 여러 Query head 가 **하나의 KV head 를 공유**합니다(group 크기 `g = H / H_kv`).
같은 group 안의 head 들을 나란히 놓으면, K/V 를 공유하기 때문에 **거시 패턴(밝은 위치)이 서로 닮아** 있습니다.
이것이 성능을 유지하면서 KV 캐시를 `g` 배 줄이는 비결입니다.

In [ ]:
# 5-1. GQA: Qwen2.5-1.5B-Instruct
free(model_mha)  # MHA 모델 내리고
tok_gqa, model_gqa = load_model("Qwen/Qwen2.5-1.5B-Instruct")
cfg = model_gqa.config
g = kv_group_size(cfg)
print(f"GQA Qwen2.5-1.5B | layers={cfg.num_hidden_layers} heads={cfg.num_attention_heads} "
      f"kv_heads={cfg.num_key_value_heads} group={g}")

toks_gqa, atts_gqa = get_attentions(tok_gqa, model_gqa, DEMO_TEXT)
L = cfg.num_hidden_layers

# 같은 KV group(group 0) 의 head 들을 나란히 → 서로 닮은 패턴
grp0 = list(range(0, min(g, 4)))
plot_heads(atts_gqa, layer=L // 2, heads=grp0, toks=toks_gqa, group_size=g,
           title=f"GQA — layer {L//2}: 같은 KV group(0) head 들은 패턴이 닮음")

# 비교용: 서로 다른 group 의 head 1개씩 → 패턴이 다름
diff = [0, g, 2 * g, 3 * g]
diff = [h for h in diff if h < cfg.num_attention_heads][:4]
plot_heads(atts_gqa, layer=L // 2, heads=diff, toks=toks_gqa, group_size=g,
           title=f"GQA — layer {L//2}: 다른 KV group 의 head 들은 패턴이 다름")

## 6. MLA — latent 압축에도 head 는 살아있다 (DeepSeek-V2-Lite)

MLA 는 K/V 를 저랭크 latent `c_KV`(차원 `kv_lora_rank`) + decoupled RoPE key 로 **압축 저장**합니다.
저장은 압축돼 있지만, 어텐션 계산 시 각 head 별로 복원·결합되므로 **head 별 패턴은 MHA 처럼 다양하게** 나타납니다.

> **A100 필요**: bf16 기준 ~31GB. OOM 이면 이 섹션을 건너뛰고 7장(footprint)으로 가세요 — footprint 비교는 모델 로드 없이 동작합니다.

In [ ]:
# 6-1. MLA: DeepSeek-V2-Lite  (A100 권장)
try:
    free(model_gqa)
    tok_mla, model_mla = load_model("deepseek-ai/DeepSeek-V2-Lite")
    cfg = model_mla.config
    print(f"MLA DeepSeek-V2-Lite | layers={cfg.num_hidden_layers} heads={cfg.num_attention_heads} "
          f"kv_lora_rank={getattr(cfg,'kv_lora_rank',None)} qk_rope={getattr(cfg,'qk_rope_head_dim',None)}")
    toks_mla, atts_mla = get_attentions(tok_mla, model_mla, DEMO_TEXT)
    L = cfg.num_hidden_layers
    plot_heads(atts_mla, layer=L // 2, heads=[0, 1, 2, 3], toks=toks_mla,
               title=f"MLA (DeepSeek-V2-Lite) — layer {L//2}: latent 압축에도 head 별 패턴 유지")
    MLA_OK = True
except Exception as e:
    print("MLA 섹션 건너뜀:", repr(e)[:200])
    print("→ 7장 KV footprint 비교는 그대로 진행됩니다.")
    MLA_OK = False

## 7. KV Cache Footprint — 토큰당 메모리와 컨텍스트 폭발

어텐션 맵이 *질적* 차이라면, KV footprint 는 *양적* 차이입니다.
**공정 비교를 위해 동일한 7B급 레퍼런스 구조**에 세 KV 공식을 적용합니다(모델 크기 차이로 인한 왜곡 제거).

- MHA: `2 · L · H · d · bytes`
- GQA: `2 · L · (H/g) · d · bytes`
- MLA: `L · (r + d_rope) · bytes`  *(압축 joint latent + decoupled RoPE key, ×2 아님)*

In [ ]:
# 7-1. 동일 7B급 레퍼런스 config 로 apples-to-apples 비교
REF = dict(num_layers=32, num_q_heads=32, head_dim=128,
           gqa_kv_heads=8,          # group = 4
           mla_kv_lora_rank=512, mla_qk_rope=64,
           dtype_bytes=2)           # bf16/fp16

def per_token_bytes(ref):
    L, H, d, b = ref["num_layers"], ref["num_q_heads"], ref["head_dim"], ref["dtype_bytes"]
    mha = 2 * L * H * d * b
    gqa = 2 * L * ref["gqa_kv_heads"] * d * b
    mla = L * (ref["mla_kv_lora_rank"] + ref["mla_qk_rope"]) * b
    return {"MHA": mha, "GQA": gqa, "MLA": mla}

pt = per_token_bytes(REF)
print("토큰당 KV cache (KB):", {k: round(v / 1024, 2) for k, v in pt.items()})
print(f"압축비  MHA→GQA = {pt['MHA']/pt['GQA']:.1f}x,  MHA→MLA = {pt['MHA']/pt['MLA']:.1f}x")

In [ ]:
# 7-2. 컨텍스트 길이별 총 KV 메모리 (배치 1 기준)
seq_lengths = np.array([6, 128, 1024, 4096, 32768])   # trace 6토큰 → long context
mem = {k: pt[k] * seq_lengths / (1024**2) for k in pt}  # MB

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
w = 0.25; x = np.arange(len(seq_lengths))
for i, (k, c) in enumerate(zip(["MHA", "GQA", "MLA"], ["salmon", "skyblue", "lightgreen"])):
    ax[0].bar(x + (i - 1) * w, mem[k], w, label=k, color=c)
ax[0].set_yscale("log")
ax[0].set_xticks(x); ax[0].set_xticklabels([f"{s}" for s in seq_lengths])
ax[0].set_xlabel("context length (tokens)"); ax[0].set_ylabel("KV cache (MB, log)")
ax[0].set_title("KV Cache Footprint (7B-class, batch=1)"); ax[0].legend(); ax[0].grid(axis="y", alpha=.3)

for k, c in zip(["MHA", "GQA", "MLA"], ["salmon", "skyblue", "lightgreen"]):
    ax[1].plot(seq_lengths, mem[k], marker="o", label=k, color=c)
ax[1].set_xlabel("context length (tokens)"); ax[1].set_ylabel("KV cache (MB)")
ax[1].set_title("기울기 = long context 에서의 메모리 증가율"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

import pandas as pd
df = pd.DataFrame({"context": seq_lengths,
                   "MHA (MB)": mem["MHA"].round(1),
                   "GQA (MB)": mem["GQA"].round(1),
                   "MLA (MB)": mem["MLA"].round(2)})
display(df)

In [ ]:
# 7-3. (참고) 실제 로드한 모델 config 에서 직접 계산
def kv_footprint_from_config(cfg, dtype_bytes=2):
    L  = cfg.num_hidden_layers
    nh = cfg.num_attention_heads
    hd = getattr(cfg, "head_dim", None) or (cfg.hidden_size // nh)
    if getattr(cfg, "kv_lora_rank", None):                      # MLA
        rope = getattr(cfg, "qk_rope_head_dim", 0)
        return L * (cfg.kv_lora_rank + rope) * dtype_bytes, "MLA"
    nkv = getattr(cfg, "num_key_value_heads", nh) or nh
    kind = "MHA" if nkv == nh else "GQA"
    return 2 * L * nkv * hd * dtype_bytes, kind

rows = []
for name in ["gpt2", "Qwen/Qwen2.5-1.5B-Instruct", "deepseek-ai/DeepSeek-V2-Lite"]:
    try:
        c = AutoConfig.from_pretrained(name, trust_remote_code=True)
        b, kind = kv_footprint_from_config(c)
        rows.append({"model": name, "kind": kind,
                     "per_token_KB": round(b / 1024, 2),
                     "@4K_MB": round(b * 4096 / 1024**2, 1)})
    except Exception as e:
        rows.append({"model": name, "kind": "?", "per_token_KB": None, "@4K_MB": repr(e)[:40]})
display(pd.DataFrame(rows))

## 8. 정리 — 청중 전달 스토리라인

1. **MHA**: head 들이 완전히 독립(맵이 제각각) → 표현력은 좋지만 KV 캐시가 가장 큼.
2. **GQA**: 같은 KV group head 들이 닮은 패턴(맵) → 성능 방어하면서 KV `g` 배 절감.
3. **MLA**: latent 압축 저장(맵은 여전히 다양) → KV 를 극단적으로 줄여 long context 를 가능하게.

### 메인 튜토리얼과의 연결
- **KV footprint 절감(GQA/MLA)** 과 **KV 재사용(prefix caching)** 은 메모리 장벽을 푸는 *두 축*입니다.
- footprint 를 줄이면 같은 VRAM 에 **더 긴 컨텍스트 · 더 많은 동시 요청**을 담을 수 있고,
  prefix caching(`cached_tokens`)은 그 캐시를 **재계산 없이 재사용**합니다.
- 둘이 합쳐져야 Mooncake / KVCache-in-the-Wild 가 말하는 *대규모 서빙 효율* 이 완성됩니다.

### 더 해보기
- `DEMO_TEXT` 를 바꿔(더 긴 반복 prefix) induction head 패턴을 더 뚜렷하게 관찰.
- 레이어 인덱스를 바꿔(초반/후반) 레이어별 어텐션 역할 차이 비교.
- `REF` config 를 본인 타깃 모델 스펙으로 바꿔 footprint 재계산.
